# ClinicalBERT -- Biological Pathways x NANDA-I Nursing Diagnoses

**Project:** Translational bioinformatics framework -- Ischemic Stroke -> NANDA-I  
**Goal:** Use ClinicalBERT to semantically map enriched biological pathways from
ischemic stroke transcriptomics to standardized NANDA-I nursing diagnoses,
building a quantitative bridge between the patient's transcriptome and
bedside nursing care planning.

### Inputs
- `string_enrichment/STRING_enrichment_Up_all.tsv` -- enriched pathways for up-regulated genes
- `string_enrichment/STRING_enrichment_Down_all.tsv` -- enriched pathways for down-regulated genes

Both files are generated by `script.R` and must exist before running this notebook.

### How to run
Execute cells in order with **Shift+Enter** or the play button.  
Outputs are saved to `outputs/figures/` (heatmaps) and `outputs/nanda/` (CSV tables).


In [ ]:
# -- Cell 1: Install dependencies ------------------------------------------
# Run once per session. In Colab this typically takes 2-3 minutes.

# ============================================================
# CÉLULA 1 — Instalar dependências
# Tempo estimado: 2-3 minutos
# ============================================================

%pip install transformers torch pandas numpy scikit-learn seaborn matplotlib sentence-transformers -q
print('\u2713 Dependências instaladas')

In [ ]:
# -- Cell 1b: HuggingFace authentication ----------------------------------
# Authenticate with HuggingFace Hub to avoid rate limits during model
# download. Get your free token at: https://huggingface.co/settings/tokens
# (Read access is sufficient for public models like ClinicalBERT)

# ============================================================
# CÉLULA 1b — Autenticação HuggingFace Hub
# Cole seu token abaixo (acesso de leitura é suficiente)
# Obtenha em: https://huggingface.co/settings/tokens
# ============================================================

import os
from huggingface_hub import login

HF_TOKEN = " "  # <-- cole seu token aqui: hf_...

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✓ HuggingFace autenticado com sucesso')
else:
    import warnings
    warnings.filterwarnings('ignore', message='.*unauthenticated.*')
    print('⚠️  HF_TOKEN não definido — usando acesso anônimo (rate limits reduzidos)')
    print('   Para autenticar, insira seu token em HF_TOKEN acima.')

In [ ]:
# -- Cell 2: Load ClinicalBERT (CLS extraction) + S-PubMedBert (SBERT) ----
# ClinicalBERT (medicalai/ClinicalBERT) is a BERT model pre-trained on
# clinical notes from the MIMIC-III dataset.
# Strategy: extract the [CLS] token vector (last_hidden_state[:, 0, :]),
# which aggregates sentence-level context without mean-pooling dilution.
# S-PubMedBert (pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT) is a biomedical
# sentence-transformer fine-tuned for semantic similarity, complementing
# the clinical-context signal from ClinicalBERT.

# ============================================================
# CÉLULA 2 — Importar bibliotecas e carregar modelos
# Estratégia híbrida: [CLS] ClinicalBERT + SBERT S-PubMedBert
# Tempo estimado: 2-4 minutos (download automático ~800MB total)
# ============================================================

from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer, util
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
import os
from IPython.display import display

# ── ClinicalBERT via [CLS] ───────────────────────────────────────────────
print('Carregando ClinicalBERT (extração via [CLS])...')
tokenizer_cls = AutoTokenizer.from_pretrained('medicalai/ClinicalBERT')
model_cls     = AutoModel.from_pretrained('medicalai/ClinicalBERT')
model_cls.eval()

def get_cls_embedding(texts, batch_size=32):
    """Extrai o vetor do token [CLS] do ClinicalBERT, normalizado L2.
    Extrae o vector del token [CLS] de ClinicalBERT, normalizado L2.
    Extract the [CLS] token vector from ClinicalBERT, L2-normalized.
    """
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        encoded = tokenizer_cls(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )
        with torch.no_grad():
            output = model_cls(**encoded)
        # [CLS] token = posição 0 do último estado oculto
        # [CLS] token = position 0 of the last hidden state
        cls_vec = output.last_hidden_state[:, 0, :]  # (batch, 768)
        cls_vec = F.normalize(cls_vec, p=2, dim=1)   # normalização L2
        all_embeddings.append(cls_vec.cpu().numpy())
    return np.vstack(all_embeddings)

print('\u2713 ClinicalBERT carregado (extração via [CLS])')

# ── S-PubMedBert (SBERT) ────────────────────────────────────────────────
print('Carregando S-PubMedBert (similaridade semântica)...')
# model_sbert = SentenceTransformer('pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT')
model_sbert = SentenceTransformer('outputs/sbert_nanda_finetuned')
print('\u2713 S-PubMedBert carregado (similaridade semântica)')

In [ ]:
# -- Cell 3: Load enriched biological pathways ------------------------------
# We read the STRING enrichment results produced by script.R for both
# up-regulated and down-regulated gene sets.
#
# Only biologically interpretable annotation categories are kept.
# PMID entries (literature co-citations with paper titles as descriptions)
# and network-topology annotations are excluded.
#
# The two gene sets represent distinct stroke phenomena:
#   Up genes  -> innate immune activation (neutrophil degranulation, cytokines)
#   Down genes -> post-stroke immunodepression (lymphocyte suppression)

# ============================================================
# CÉLULA 3 — Carregar vias enriquecidas (Up E Down)
# Rastreabilidade total R → Python para ambas as direções
# ============================================================

TSV_UP   = "string_enrichment/STRING_enrichment_Up_all.tsv"
TSV_DOWN = "string_enrichment/STRING_enrichment_Down_all.tsv"

for path in [TSV_UP, TSV_DOWN]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Arquivo não encontrado: {path}\n"
            "Execute script.R até a seção de exportação STRINGdb antes de rodar este notebook."
        )

# Categorias funcionais — excluem PMID (literatura) e NetworkNeighborAL
CATEGORIAS_VALIDAS = ["Process", "RCTM", "Component", "Keyword", "COMPARTMENTS",
                      "InterPro", "SMART", "HPO", "Pfam"]

def carregar_vias(path, regulacao):
    enr = pd.read_csv(path, sep="\t")
    curado = enr[
        enr["category"].isin(CATEGORIAS_VALIDAS) & (enr["fdr"] < 0.05)
    ].copy()
    curado["regulacao"] = regulacao
    return curado.sort_values("fdr").reset_index(drop=True)

enr_up_df   = carregar_vias(TSV_UP,   "Up")
enr_down_df = carregar_vias(TSV_DOWN, "Down")

# Listas de descrições (sem duplicatas, mantendo ordem por FDR)
vias_up   = list(dict.fromkeys(enr_up_df["description"].tolist()))
vias_down = list(dict.fromkeys(enr_down_df["description"].tolist()))

print(f"✓ Vias carregadas de arquivos STRING:")
print(f"\n  ► Up ({len(vias_up)} termos)  —  {TSV_UP}")
for i, v in enumerate(vias_up, 1):
    fdr = enr_up_df[enr_up_df['description'] == v]['fdr'].values[0]
    cat = enr_up_df[enr_up_df['description'] == v]['category'].values[0]
    print(f"    {i:2d}. [{cat}] {v}  (FDR={fdr:.2e})")

print(f"\n  ► Down ({len(vias_down)} termos)  —  {TSV_DOWN}")
for i, v in enumerate(vias_down, 1):
    fdr = enr_down_df[enr_down_df['description'] == v]['fdr'].values[0]
    cat = enr_down_df[enr_down_df['description'] == v]['category'].values[0]
    print(f"    {i:2d}. [{cat}] {v}  (FDR={fdr:.2e})")

In [ ]:
# -- Cell 4: NANDA-I Nursing Diagnoses ---------------------------------------
# NANDA International (NANDA-I) is the globally recognized taxonomy for
# standardized nursing diagnoses. Each entry is a formally validated
# diagnosis with its official code and a clinical description written in
# language that ClinicalBERT was trained on, maximizing semantic alignment.
#
# Diagnoses were selected for clinical relevance to ischemic stroke:
# neurological impairment, immune dysfunction, mobility deficits,
# safety risks, and psychosocial responses to disability.

# ============================================================
# CÉLULA 4 — Diagnósticos de Enfermagem NANDA-I
# Taxonomia NANDA-I — linguagem clínica padronizada internacionalmente
# Diagnósticos selecionados por relevância clínica para AVC isquêmico
# ============================================================

nanda_diagnosticos = [
    # ── Domínio 2: Nutrição ──────────────────────────────────────────────────
    "Imbalanced nutrition less than body requirements: inadequate nutritional intake related to dysphagia and impaired swallowing after stroke (NANDA-I 00002)",
    "Impaired swallowing: abnormal functioning of swallowing mechanism compromising nutrition and airway protection due to neurological deficit (NANDA-I 00103)",
    "Risk for aspiration: susceptibility to entry of secretions into tracheobronchial passages due to impaired swallowing and reduced consciousness (NANDA-I 00039)",

    # ── Domínio 3: Eliminação e Troca ────────────────────────────────────────
    "Impaired urinary elimination: dysfunction of urinary elimination related to neurogenic bladder dysfunction after stroke (NANDA-I 00016)",
    "Constipation: decrease in normal bowel movement frequency and consistency related to immobility and neurological dysfunction (NANDA-I 00011)",
    "Impaired gas exchange: excess or deficit in oxygenation and carbon dioxide elimination related to ventilation-perfusion imbalance (NANDA-I 00030)",
    "Ineffective airway clearance: inability to clear secretions from respiratory tract due to reduced consciousness and neuromuscular dysfunction (NANDA-I 00031)",

    # ── Domínio 4: Atividade e Repouso ───────────────────────────────────────
    "Impaired physical mobility: limitation in independent purposeful physical movement related to neurological impairment and hemiplegia (NANDA-I 00085)",
    "Fatigue: overwhelming sustained sense of exhaustion and decreased capacity for physical and mental work related to neurological disease (NANDA-I 00093)",
    "Disturbed sleep pattern: time-limited disruption of sleep amount and quality related to hospitalization neurological symptoms and pain (NANDA-I 00198)",
    "Bathing self-care deficit: impaired ability to independently perform bathing and hygiene activities related to hemiplegia and motor deficit (NANDA-I 00108)",
    "Dressing self-care deficit: impaired ability to dress and undress independently related to motor and sensory deficit after stroke (NANDA-I 00109)",
    "Risk for disuse syndrome: susceptibility to deterioration of body systems related to prescribed or unavoidable musculoskeletal inactivity (NANDA-I 00040)",
    "Decreased cardiac output: inadequate quantity of blood pumped by heart to meet metabolic demands of body related to cardiac dysfunction (NANDA-I 00029)",

    # ── Domínio 5: Percepção e Cognição ──────────────────────────────────────
    "Acute confusion: abrupt onset of reversible disturbances of consciousness attention cognition and perception related to neuroinflammation and cerebral ischemia (NANDA-I 00128)",
    "Impaired verbal communication: decreased ability to receive process transmit and use a system of symbols related to aphasia after stroke (NANDA-I 00051)",
    "Impaired memory: inability to recall or retrieve bits of information or behavioral skills related to neurological damage from cerebral ischemia (NANDA-I 00131)",
    "Decreased intracranial adaptive capacity: intracranial fluid dynamic mechanisms compromised resulting in repeated disproportionate increases in intracranial pressure (NANDA-I 00049)",

    # ── Domínio 6: Autopercepção ──────────────────────────────────────────────
    "Hopelessness: subjective state in which individual sees limited or no alternatives or personal choices and is unable to mobilize energy related to neurological disability (NANDA-I 00124)",
    "Powerlessness: lived experience of lack of control over a situation including a perception that one's actions do not significantly affect outcome related to stroke-induced dependency (NANDA-I 00125)",
    "Disturbed body image: confusion in mental picture of one's physical self related to functional and physical loss after stroke and hemiplegia (NANDA-I 00118)",

    # ── Domínio 7: Papéis e Relacionamentos ──────────────────────────────────
    "Caregiver role strain: difficulty in performing the family caregiver role related to complexity duration and demands of post-stroke care (NANDA-I 00061)",
    "Interrupted family processes: change in family relationships and functioning related to acute illness and hospitalization of family member (NANDA-I 00060)",

    # ── Domínio 9: Enfrentamento e Tolerância ao Estresse ────────────────────
    "Anxiety: vague uneasy feeling of discomfort or dread accompanied by autonomic response related to neurological deficit uncertainty and loss of independence (NANDA-I 00146)",
    "Fear: response to perceived threat that is consciously recognized as a danger related to disability dependency and uncertain prognosis (NANDA-I 00148)",
    "Grieving: functional process that includes emotional physical spiritual social and intellectual responses related to loss of neurological function and previous lifestyle (NANDA-I 00136)",

    # ── Domínio 11: Segurança e Proteção ─────────────────────────────────────
    "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
    "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
    "Risk for injury: susceptibility to physical damage due to sensory and motor deficit impaired balance and altered consciousness after stroke (NANDA-I 00035)",
    "Risk for falls: susceptibility to falling that may cause physical harm and health deterioration related to neurological impairment and altered gait (NANDA-I 00155)",
    "Risk for pressure injury: susceptibility to localized damage to skin and underlying tissue usually over bony prominence related to immobility and sensory deficit (NANDA-I 00249)",
    "Hyperthermia: core body temperature above normal diurnal range related to systemic inflammatory response and post-stroke infection (NANDA-I 00007)",
    "Risk for thrombus: susceptibility to obstruction of a blood vessel by a thrombus that can cause ischemia necrosis and death related to immobility after stroke (NANDA-I 00291)",
    "Impaired tissue integrity: damage to mucous membrane corneal integumentary or subcutaneous tissues related to ischemic injury and impaired circulation (NANDA-I 00044)",

    # ── Domínio 12: Conforto ──────────────────────────────────────────────────
    "Acute pain: unpleasant sensory and emotional experience associated with actual or potential tissue damage related to neurological injury and post-stroke pain (NANDA-I 00132)",
    "Impaired comfort: perceived lack of ease relief and transcendence in physical psychospiritual environmental and social dimensions related to neurological symptoms (NANDA-I 00214)",

    # ── Domínio 4: Perfusão e Circulação ──────────────────────────────────────
    "Ineffective peripheral tissue perfusion: decrease in blood circulation to the periphery that may compromise health related to vascular dysfunction and immobility (NANDA-I 00204)",
    "Risk for ineffective cerebral tissue perfusion: susceptibility to decrease in cerebral tissue circulation related to cerebral ischemia edema and blood-brain barrier disruption (NANDA-I 00201)"
]

print(f"✓ {len(nanda_diagnosticos)} diagnósticos NANDA-I carregados")
print("\n  Distribuição por domínio:")
dominios = {
    "Nutrição (D2)"              : [d for d in nanda_diagnosticos if any(c in d for c in ["00002","00103","00039"])],
    "Eliminação/Troca (D3)"      : [d for d in nanda_diagnosticos if any(c in d for c in ["00016","00011","00030","00031"])],
    "Atividade/Repouso (D4)"     : [d for d in nanda_diagnosticos if any(c in d for c in ["00085","00093","00198","00108","00109","00040","00029"])],
    "Percepção/Cognição (D5)"    : [d for d in nanda_diagnosticos if any(c in d for c in ["00128","00051","00131","00049"])],
    "Autopercepção (D6)"         : [d for d in nanda_diagnosticos if any(c in d for c in ["00124","00125","00118"])],
    "Papéis/Relac. (D7)"         : [d for d in nanda_diagnosticos if any(c in d for c in ["00061","00060"])],
    "Enfrentamento (D9)"         : [d for d in nanda_diagnosticos if any(c in d for c in ["00146","00148","00136"])],
    "Segurança/Proteção (D11)"   : [d for d in nanda_diagnosticos if any(c in d for c in ["00004","00043","00035","00155","00249","00007","00291","00044"])],
    "Conforto (D12)"             : [d for d in nanda_diagnosticos if any(c in d for c in ["00132","00214"])],
    "Perfusão/Circulação"        : [d for d in nanda_diagnosticos if any(c in d for c in ["00204","00201"])],
}
for dom, diags in dominios.items():
    print(f"    {dom}: {len(diags)} diagnóstico(s)")

In [ ]:
# -- Cell 5: Generate embeddings (ClinicalBERT [CLS] + SBERT) -------------
# Two independent embedding spaces are produced for each text set:
#   [CLS]  captures clinical context and implicit relations
#          (causality, risk, abstraction) via the first BERT token
#   SBERT  captures sentential semantic similarity via a model
#          fine-tuned for biomedical retrieval
# Fusion happens in similarity space (Cell 6), not in vector space,
# to avoid distortion of cosine distances between heterogeneous latents.

# ============================================================
# CÉLULA 5 — Gerar embeddings separados [CLS] e SBERT
# A fusão ocorre na célula 6, no espaço de similaridade
# ============================================================

# ── Embeddings ClinicalBERT via [CLS] ───────────────────────────────────
print('Gerando embeddings [CLS] das vias Up...')
cls_up    = get_cls_embedding(vias_up)
print('Gerando embeddings [CLS] das vias Down...')
cls_down  = get_cls_embedding(vias_down)
print('Gerando embeddings [CLS] dos diagnósticos NANDA-I...')
cls_nanda = get_cls_embedding(nanda_diagnosticos)

# ── Embeddings SBERT (semântica sentencial) ──────────────────────────────
print('\nGerando embeddings SBERT das vias Up...')
sbert_up    = model_sbert.encode(vias_up,            convert_to_numpy=True, show_progress_bar=True)
print('Gerando embeddings SBERT das vias Down...')
sbert_down  = model_sbert.encode(vias_down,          convert_to_numpy=True, show_progress_bar=True)
print('Gerando embeddings SBERT dos diagnósticos NANDA-I...')
sbert_nanda = model_sbert.encode(nanda_diagnosticos, convert_to_numpy=True, show_progress_bar=True)

print(f'\n✓ Embeddings gerados (sem fusão — combinação ocorre na célula 6)')
print(f'  [CLS] ClinicalBERT : dim={cls_up.shape[1]}')
print(f'  SBERT S-PubMedBert : dim={sbert_up.shape[1]}')
print(f'  Vias Up            : {cls_up.shape[0]} vetores em cada espaço')
print(f'  Vias Down          : {cls_down.shape[0]} vetores em cada espaço')
print(f'  Diagnósticos NANDA-I : {cls_nanda.shape[0]} vetores em cada espaço')

In [ ]:
# -- Cell 6: Cosine similarity matrices (late fusion) ----------------------
# Similarity matrices are computed independently in each embedding space
# and then combined by weighted average (late fusion).
# This avoids distorting cosine distances that would occur when
# concatenating heterogeneous latent vectors before similarity computation.
#
#   sim_final = ALPHA * sim([CLS]) + (1 - ALPHA) * sim(SBERT)
#
# Two separate matrices are produced:
#   df_matriz_up   -- Up pathways   x NANDA-I diagnoses
#   df_matriz_down -- Down pathways x NANDA-I diagnoses

# ============================================================
# CÉLULA 6 — Fusão tardia no espaço de similaridade de cosseno
# Uma matriz por direção regulatória
# ============================================================

ALPHA = 0.5  # peso do ClinicalBERT [CLS]; (1 - ALPHA) = peso do SBERT
             # weight of ClinicalBERT [CLS]; (1 - ALPHA) = SBERT weight

# ── Up × NANDA-I ─────────────────────────────────────────────────────────
sim_cls_up   = util.cos_sim(
    torch.tensor(cls_up),
    torch.tensor(cls_nanda)
).numpy()

sim_sbert_up = util.cos_sim(
    torch.tensor(sbert_up),
    torch.tensor(sbert_nanda)
).numpy()

sim_up = ALPHA * sim_cls_up + (1 - ALPHA) * sim_sbert_up  # fusão ponderada / weighted fusion

df_matriz_up = pd.DataFrame(sim_up, index=vias_up, columns=nanda_diagnosticos)

# ── Down × NANDA-I ───────────────────────────────────────────────────────
sim_cls_down   = util.cos_sim(
    torch.tensor(cls_down),
    torch.tensor(cls_nanda)
).numpy()

sim_sbert_down = util.cos_sim(
    torch.tensor(sbert_down),
    torch.tensor(sbert_nanda)
).numpy()

sim_down = ALPHA * sim_cls_down + (1 - ALPHA) * sim_sbert_down  # fusão ponderada / weighted fusion

df_matriz_down = pd.DataFrame(sim_down, index=vias_down, columns=nanda_diagnosticos)

print(f'✓ Matrizes de similaridade calculadas (fusão tardia)')
print(f'  Estratégia: ALPHA={ALPHA} × sim([CLS]) + {1-ALPHA} × sim(SBERT)')
print(f'\n  Up × NANDA-I   : {df_matriz_up.shape[0]} vias × {df_matriz_up.shape[1]} diagnósticos')
print(f'    Score min/max/mean : {sim_up.min():.3f} / {sim_up.max():.3f} / {sim_up.mean():.3f}')
print(f'\n  Down × NANDA-I : {df_matriz_down.shape[0]} vias × {df_matriz_down.shape[1]} diagnósticos')
print(f'    Score min/max/mean : {sim_down.min():.3f} / {sim_down.max():.3f} / {sim_down.mean():.3f}')

In [ ]:
# -- Cell 7: Rank pathway-diagnosis pairs and apply similarity threshold ----
# For each biological pathway, we rank the top-5 most semantically similar
# NANDA-I diagnoses. Pairs below SIMILARITY_THRESHOLD (0.65) are excluded
# from the final mapping.
#
# Threshold rationale: scores below 0.65 tend to reflect shared vocabulary
# rather than genuine semantic alignment. Adjust between 0.60 and 0.75
# based on inspection of the score distributions above.

# ============================================================
# CÉLULA 7 — Ranking com limiar de similaridade por direção
# SIMILARITY_THRESHOLD: pares abaixo excluídos do mapeamento final
# Referência: 0.65 exclui co-ocorrência lexical fraca; ajuste 0.60–0.75
# ============================================================

SIMILARITY_THRESHOLD = 0.65
TOP_N = 5

def gerar_ranking(df_matriz, regulacao, top_n=TOP_N):
    resultados = []
    for via in df_matriz.index:
        scores = df_matriz.loc[via].sort_values(ascending=False).head(top_n)
        for rank, (diagnostico, sim) in enumerate(scores.items(), 1):
            resultados.append({
                'regulacao'         : regulacao,
                'via'               : via,
                'rank'              : rank,
                'diagnostico_nanda' : diagnostico,
                'similaridade'      : round(float(sim), 4)
            })
    return pd.DataFrame(resultados)

df_rank_up   = gerar_ranking(df_matriz_up,   "Up")
df_rank_down = gerar_ranking(df_matriz_down, "Down")
df_rank_all  = pd.concat([df_rank_up, df_rank_down], ignore_index=True)

# Filtro por threshold
df_validos_up   = df_rank_up[df_rank_up['similaridade']   >= SIMILARITY_THRESHOLD].copy()
df_validos_down = df_rank_down[df_rank_down['similaridade'] >= SIMILARITY_THRESHOLD].copy()
df_validos_all  = df_rank_all[df_rank_all['similaridade']  >= SIMILARITY_THRESHOLD].copy()

def imprimir_ranking(df_validos, regulacao):
    print(f"\n{'='*70}")
    print(f"  GENES {regulacao.upper()} — Pares válidos (≥ {SIMILARITY_THRESHOLD})")
    print(f"{'='*70}")
    if df_validos.empty:
        print("  Nenhum par acima do limiar.")
        return
    for via in df_validos['via'].unique():
        print(f"\n  ► {via}")
        subset = df_validos[df_validos['via'] == via]
        for _, row in subset.iterrows():
            nivel  = "FORTE" if row['similaridade'] >= 0.80 else "ALTO" if row['similaridade'] >= 0.70 else "MODERADO"
            codigo = re.search(r'NANDA-I (\d+)', row['diagnostico_nanda'])
            cod    = f"[{codigo.group(1)}]" if codigo else ""
            label  = re.sub(r':.*', '', re.sub(r'\s*\(NANDA-I \d+\)', '', row['diagnostico_nanda']))[:55]
            print(f"    {int(row['rank'])}. [{nivel}] {cod} {label} ({row['similaridade']:.4f})")

imprimir_ranking(df_validos_up,   "Up")
imprimir_ranking(df_validos_down, "Down")

print(f"\n{'='*70}")
print(f"  RESUMO")
print(f"{'='*70}")
print(f"  Up   — pares válidos : {len(df_validos_up)} / {len(df_rank_up)}")
print(f"  Down — pares válidos : {len(df_validos_down)} / {len(df_rank_down)}")
print(f"  Total válidos        : {len(df_validos_all)}")

In [ ]:
# -- Cell 8: Heatmaps -------------------------------------------------------
# One heatmap per regulatory direction. Color encodes cosine similarity
# (warm = high semantic alignment; cool = low). Labels are shortened for
# readability; full descriptions are preserved in the exported CSVs.

# ============================================================
# CÉLULA 8 — Heatmaps por direção regulatória
# ============================================================

def abreviar(texto, max_len=42):
    """Remove código NANDA e abrevia para exibição no heatmap."""
    texto_limpo = re.sub(r'\s*\(NANDA-I \d+\)', '', texto)
    texto_limpo = re.sub(r':.*', '', texto_limpo)  # mantém só o rótulo principal
    return texto_limpo[:max_len] + '…' if len(texto_limpo) > max_len else texto_limpo

def plotar_heatmap(df_matriz, titulo, nome_arquivo):
    df_heat = df_matriz.copy()
    df_heat.index   = [abreviar(v, 38) for v in df_heat.index]
    df_heat.columns = [abreviar(c, 40) for c in df_heat.columns]

    n_vias  = len(df_heat)
    n_nanda = len(df_heat.columns)
    altura  = max(5,  n_vias  * 0.65 + 3)
    largura = max(22, n_nanda * 0.52 + 4)

    fig, ax = plt.subplots(figsize=(largura, altura))
    sns.heatmap(
        df_heat,
        annot      = True,
        fmt        = '.2f',
        cmap       = 'RdYlBu_r',
        center     = 0.5,
        vmin       = 0.30,
        vmax       = 1.00,
        linewidths = 0.3,
        ax         = ax,
        annot_kws  = {'size': 6}
    )
    ax.set_title(titulo, fontsize=12, fontweight='bold', pad=14)
    ax.set_xlabel('Diagnósticos de Enfermagem NANDA-I', fontsize=9)
    ax.set_ylabel('Vias Biológicas Enriquecidas (STRING)', fontsize=9)
    ax.tick_params(axis='x', rotation=55, labelsize=6.5)
    ax.tick_params(axis='y', rotation=0,  labelsize=7.5)
    plt.tight_layout()
    plt.savefig(nome_arquivo, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✓ Heatmap salvo: {nome_arquivo}')

plotar_heatmap(
    df_matriz_up,
    'ClinicalBERT — Vias Up-reguladas × Diagnósticos NANDA-I\n'
    '(Genes superexpressos no AVC isquêmico — resposta inflamatória inata / neutrófilos)',
    'outputs/figures/heatmap_nanda_up.png'
)

plotar_heatmap(
    df_matriz_down,
    'ClinicalBERT — Vias Down-reguladas × Diagnósticos NANDA-I\n'
    '(Genes subexpressos no AVC isquêmico — imunossupressão / linfócitos)',
    'outputs/figures/heatmap_nanda_down.png'
)

In [ ]:
# -- Cell 9: Export results -------------------------------------------------
# Three tiers of results are exported:
#   1. Valid pairs (similarity >= threshold) -- primary scientific output
#   2. Full top-5 rankings without threshold -- for calibration and audit
#   3. Raw similarity matrices -- for custom downstream analysis

import os
os.makedirs('outputs/figures', exist_ok=True)
os.makedirs('outputs/nanda',   exist_ok=True)

# ============================================================
# CÉLULA 9 — Exportar resultados do mapeamento NANDA-I
# ============================================================

# Mapeamentos com threshold aplicado (arquivo principal para análise)
df_validos_up.to_csv('outputs/nanda/nanda_mapping_up_threshold65.csv',    index=False, encoding='utf-8')
df_validos_down.to_csv('outputs/nanda/nanda_mapping_down_threshold65.csv', index=False, encoding='utf-8')
df_validos_all.to_csv('outputs/nanda/nanda_mapping_completo_threshold65.csv', index=False, encoding='utf-8')

# Rankings completos top-5 por via (sem threshold — para auditoria e ajuste)
df_rank_all.to_csv('outputs/nanda/nanda_ranking_top5_completo.csv', index=False, encoding='utf-8')

# Matrizes completas por direção
df_matriz_up.to_csv('outputs/nanda/matriz_similaridade_up_nanda.csv',   encoding='utf-8')
df_matriz_down.to_csv('outputs/nanda/matriz_similaridade_down_nanda.csv', encoding='utf-8')

# Compatibilidade retroativa com script.R (importação pela seção 2ª etapa)
df_rank_all.rename(columns={'diagnostico_nanda': 'condicao'}).to_csv(
    'outputs/nanda/top5_condicoes_por_via.csv', index=False, encoding='utf-8'
)
df_matriz_up.to_csv('outputs/nanda/matriz_similaridade_completa.csv', encoding='utf-8')

print('✓ Arquivos exportados:')
print(f'  → nanda_mapping_up_threshold65.csv         ({len(df_validos_up)} pares válidos Up)')
print(f'  → nanda_mapping_down_threshold65.csv       ({len(df_validos_down)} pares válidos Down)')
print(f'  → nanda_mapping_completo_threshold65.csv   ({len(df_validos_all)} pares válidos total)')
print(f'  → nanda_ranking_top5_completo.csv          ({len(df_rank_all)} pares totais)')
print(f'  → matriz_similaridade_up_nanda.csv         ({df_matriz_up.shape[0]}×{df_matriz_up.shape[1]})')
print(f'  → matriz_similaridade_down_nanda.csv       ({df_matriz_down.shape[0]}×{df_matriz_down.shape[1]})')
print(f'  → outputs/figures/heatmap_nanda_up.png')
print(f'  → outputs/figures/heatmap_nanda_down.png')
print()
print('PRÓXIMO PASSO: importe outputs/nanda/nanda_mapping_completo_threshold65.csv no R para visualização e validação.')

In [ ]:
# -- Cell 10: Download outputs (Google Colab only) --------------------------
# Triggers automatic file downloads in Colab. In a local Jupyter environment
# the files are already saved to disk and this cell prints their location.

# ============================================================
# CÉLULA 10 — Download automático dos arquivos (Colab)
# ============================================================

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("⚠️  Ambiente não-Colab: downloads automáticos desativados.")
    print("   Arquivos disponíveis em:", os.getcwd())

if IN_COLAB:
    for fname in [
        'outputs/nanda/nanda_mapping_completo_threshold65.csv',
        'outputs/nanda/nanda_mapping_up_threshold65.csv',
        'outputs/nanda/nanda_mapping_down_threshold65.csv',
        'outputs/nanda/nanda_ranking_top5_completo.csv',
        'outputs/nanda/matriz_similaridade_up_nanda.csv',
        'outputs/nanda/matriz_similaridade_down_nanda.csv',
        'outputs/figures/heatmap_nanda_up.png',
        'outputs/figures/heatmap_nanda_down.png',
    ]:
        files.download(fname)
    print('✓ Downloads iniciados')

## Importing results in R

After the notebook finishes, import the NANDA-I mapping tables in R:

```r
library(dplyr)

# Primary output: valid pathway-diagnosis pairs (similarity >= 0.65)
nanda <- read.csv('outputs/nanda/nanda_mapping_completo_threshold65.csv',
                  fileEncoding = 'UTF-8')

# Filter by regulatory direction
nanda_up   <- nanda[nanda$regulacao == 'Up',   ]
nanda_down <- nanda[nanda$regulacao == 'Down', ]

# Full cosine similarity matrices for custom visualization
mat_up   <- read.csv('outputs/nanda/matriz_similaridade_up_nanda.csv',
                     row.names = 1, fileEncoding = 'UTF-8')
mat_down <- read.csv('outputs/nanda/matriz_similaridade_down_nanda.csv',
                     row.names = 1, fileEncoding = 'UTF-8')
```


In [ ]:
# -- Cell 13: Verification --------------------------------------------------
# Run this cell after all others to confirm the pipeline completed correctly.
# All assertions must pass before sharing or publishing results.

# ============================================================
# CÉLULA 13 — Verificação final (C1–C5)
# C1-C5: pipeline usa fusão tardia de similaridade [CLS] + SBERT
# ============================================================

# C1 — Vias carregadas corretamente (sem títulos de artigos)
assert isinstance(vias_up,   list) and len(vias_up)   > 0, "C1: vias_up vazia"
assert isinstance(vias_down, list) and len(vias_down) > 0, "C1: vias_down vazia"
assert all(len(v) < 120 for v in vias_up + vias_down), \
    "C1: possíveis títulos de artigos detectados (strings > 120 chars)"

# C2 — Diagnósticos NANDA-I presentes e com código
assert len(nanda_diagnosticos) >= 30, "C2: menos de 30 diagnósticos NANDA-I carregados"
assert all("NANDA-I" in d for d in nanda_diagnosticos), \
    "C2: algum diagnóstico sem código NANDA-I"

# C3 — Threshold aplicado corretamente
assert 'SIMILARITY_THRESHOLD' in dir(), "C3: SIMILARITY_THRESHOLD não definido"
if len(df_validos_all) > 0:
    assert df_validos_all['similaridade'].min() >= SIMILARITY_THRESHOLD, \
        f"C3: pares abaixo do threshold {SIMILARITY_THRESHOLD} encontrados"

# C4 — Arquivos exportados
arquivos_esperados = [
    'outputs/nanda/nanda_mapping_completo_threshold65.csv',
    'outputs/nanda/matriz_similaridade_up_nanda.csv',
    'outputs/nanda/matriz_similaridade_down_nanda.csv',
    'outputs/figures/heatmap_nanda_up.png',
    'outputs/figures/heatmap_nanda_down.png',
]
for arq in arquivos_esperados:
    assert os.path.exists(arq), f"C4: arquivo não encontrado: {arq}"


# C5 — Scores de similaridade dentro do intervalo esperado [-1, 1]
# Similarity scores must lie within the valid cosine range
assert sim_up.min()   >= -1.0 and sim_up.max()   <= 1.0, "C5: sim_up fora de [-1,1]"
assert sim_down.min() >= -1.0 and sim_down.max() <= 1.0, "C5: sim_down fora de [-1,1]"

print("✅ Todas as verificações C1–C5 passaram.")
print(f"   Vias Up         : {len(vias_up)}")
print(f"   Vias Down       : {len(vias_down)}")
print(f"   NANDA-I         : {len(nanda_diagnosticos)} diagnósticos")
print(f"   Threshold       : {SIMILARITY_THRESHOLD}")
print(f"   Pares válidos Up   : {len(df_validos_up)}")
print(f"   Pares válidos Down : {len(df_validos_down)}")
print(f"   Pares válidos total: {len(df_validos_all)}")